In [2]:
import joblib
import pandas as pd
import numpy as np
import polars as pl

import yfinance as yf

import akshare as ak

In [3]:
import sys
sys.path.append('./Code')

In [4]:
sys.path.append('/home/jun23/workspace/GitHub/kzz')

In [5]:
from mytools.util import unset_proxy
from data_center import DataCenter

In [6]:
dc = DataCenter()

In [7]:
from YahooOHLCFetcher import get_adjusted_ohlc

In [55]:
olhc_data = joblib.load('./Data/sp500_ohlc_data.pkl')

In [76]:
failed_tickers = []
for ticker, df in olhc_data.items():
    tmp = df.loc[~df['close'].isna()]
    if len(tmp) == 0:
        failed_tickers.append(ticker)

In [77]:
len(failed_tickers)

150

In [78]:
failed_tickers

['ANSS',
 'BRK.B',
 'BF.B',
 'DFS',
 'HES',
 'JNPR',
 'PARA',
 'WBA',
 'GPS',
 'KSU',
 'MXIM',
 'ALXN',
 'HFC',
 'FLIR',
 'VAR',
 'CXO',
 'TIF',
 'NBL',
 'ETFC',
 'ADS',
 'JWN',
 'AGN',
 'RTN',
 'ARNC',
 'XEC',
 'WCG',
 'VIAB',
 'CELG',
 'TSS',
 'APC',
 'RHT',
 'LLL',
 'DWDP',
 'SRCL',
 'XL',
 'GGP',
 'DPS',
 'MON',
 'WYN',
 'PDCO',
 'CHK',
 'SNI',
 'BCR',
 'LVLT',
 'SPLS',
 'WFM',
 'MNK',
 'RAI',
 'YHOO',
 'MJN',
 'DNB',
 'SWN',
 'FTR',
 'LLTC',
 'ENDP',
 'STJ',
 'LM',
 'DO',
 'TYC',
 'CPGX',
 'CVC',
 'BXLT',
 'ARG',
 'TWC',
 'ESV',
 'GMCR',
 'BRCM',
 'ALTR',
 'CMCSK',
 'CSC',
 'SIAL',
 'HCBK',
 'JOY',
 'HSP',
 'PLL',
 'DTV',
 'FDO',
 'KRFT',
 'QEP',
 'LO',
 'WIN',
 'DNR',
 'AVP',
 'CFN',
 'SWY',
 'RDC',
 'X',
 'FRX',
 'IGT',
 'LSI',
 'WPX',
 'LIFE',
 'JDSU',
 'JCP',
 'NYX',
 'SAI',
 'APOL',
 'DF',
 'BIG',
 'FII',
 'RRD',
 'KFT',
 'LXK',
 'PGN',
 'NVLS',
 'AKS',
 'MWW',
 'JNS',
 'CEPH',
 'NOVL',
 'GENZ',
 'MDP',
 'EK',
 'MIL',
 'STR',
 'XTO',
 'BJS',
 'CTX',
 'ACAS',
 'LEH',
 'FRE',
 

In [66]:
start_date, end_date = '1995-01-01', '2025-10-31'
interval = '1d'

In [81]:
data, failed = get_adjusted_ohlc(
    tickers=['TWTR'],
    start_date=start_date,
    end_date=end_date,
    interval=interval
)

/home/jun23/workspace/GitHub/Cross-Sectional-Momentum-LTR/Code/YahooOHLCFetcher.py:76: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(

1 Failed download:
['TWTR']: YFTzMissingError('possibly delisted; no timezone found')



Starting download for 1 tickers in 1 batches
Date range: 1995-01-01 to 2025-10-31

Batch 1/1: Processing 1 tickers (0 remaining)
--------------------------------------------------
✗ Failed to process TWTR: Empty DataFrame for TWTR

Download Summary:
Successfully processed: 0/1 tickers
Failed tickers: ['TWTR']


# 如何获取已经退市的美股历史价格数据

## 1. yfinance

不行

In [83]:
data = yf.download(
    tickers=['TWTR'],
    start=start_date,
    end='2022-10-27',
    interval=interval,
    group_by='ticker',
    progress=False,
    threads=False,
    auto_adjust=False,
)


1 Failed download:
['TWTR']: YFTzMissingError('possibly delisted; no timezone found')


In [36]:

try:
    twtr = yf.Ticker("TWTR")
    # 明确指定获取退市前的历史数据
    hist_data = twtr.history(start="2021-01-01", end="2021-10-28")
    if hist_data.empty:
        print("未获取到数据。")
    else:
        print(hist_data.tail())
except Exception as e:
    print(f"获取数据时出现错误: {e}")

$TWTR: possibly delisted; no timezone found


未获取到数据。


## 2. Akshare

- 免费
- 按symbol查询，不能按日期

In [10]:
us_symbols = pl.DataFrame(dc.db.us_symbol.find({}, {'_id': 0}))

In [11]:
us_symbols

name,cname,symbol
str,str,str
"""NVIDIA Corp.""","""英伟达公司""","""NVDA"""
"""Apple, Inc.""","""苹果公司""","""AAPL"""
"""Microsoft Corp.""","""微软公司""","""MSFT"""
"""Alphabet, Inc.""","""谷歌""","""GOOG"""
"""Alphabet, Inc.""","""谷歌A类股""","""GOOGL"""
…,…,…
"""Zura Bio Ltd.""","""Zura Bio Ltd.""","""ZURAW"""
"""Volatility Premium Plus ETF""","""Volatility Premium Plus ETF""","""ZVOL"""
"""ZyVersa Therapeutics, Inc.""","""ZyVersa Therapeutics, Inc.""","""ZVSAW"""


In [12]:
def _ak_update_us_daily(dc, symbol):
    try:
        df = ak.stock_us_daily(symbol=symbol, adjust="")
        if len(df) == 0:
            print(f'not data for {symbol}')
            return False
        df['symbol'] = symbol
        res = dc.db.ak_us_daily.insert_many(df.to_dict('records'))
        if not res.acknowledged:
            print(f'insert to db failed, symbol={symbol}')
            return False
        return True
    except Exception as ex:
        print(f'_ak_update_us_daily throw ex: {ex}, symbol = {symbol}')
        return False

def _ak_update_us_qfq_factor(dc, symbol):
    try:
        df = ak.stock_us_daily(symbol=symbol, adjust="qfq-factor")
        if len(df) == 0:
            print(f'_ak_update_us_qfq_factor: not data for {symbol}')
            return False
        df['symbol'] = symbol
        df['qfq_factor'] = df['qfq_factor'].astype(np.float64)
        df['adjust'] = df['adjust'].astype(np.float64)
        res = dc.db.ak_us_daily_qfq_factor.insert_many(df.to_dict('records'))
        if not res.acknowledged:
            print(f'insert to db failed, symbol={symbol}')
            return False
        return True
    except Exception as ex:
        print(f'_ak_update_us_qfq_factor throw ex: {ex}, symbol = {symbol}')
        return False

In [13]:
from tqdm import tqdm

In [14]:
all_symbols = us_symbols['symbol']

In [15]:
ok = set(dc.db.ak_us_daily.distinct(key='symbol'))

In [16]:
failed_us_daily = []
failed_us_daily_qfq_factor = []

In [17]:
for symbol in tqdm(all_symbols):
    if symbol in ok:
        continue
    ret = _ak_update_us_daily(dc, symbol)
    if not ret:
        failed_us_daily.append(symbol)
        continue
    ok.add(symbol)
    ret = _ak_update_us_qfq_factor(dc, symbol)
    if not ret:
        failed_us_daily_qfq_factor.append(symbol)
    # break

 11%|████████████▏                                                                                                     | 1761/16415 [00:00<00:02, 5461.78it/s]

_ak_update_us_daily throw ex: list index out of range, symbol = EMBJ
_ak_update_us_daily throw ex: 'date', symbol = NCR
_ak_update_us_daily throw ex: 'date', symbol = AVG
_ak_update_us_daily throw ex: list index out of range, symbol = CEPV
_ak_update_us_daily throw ex: 'date', symbol = PAII.U
_ak_update_us_daily throw ex: 'date', symbol = SUNH
_ak_update_us_daily throw ex: 'date', symbol = SNLN
_ak_update_us_daily throw ex: 'date', symbol = VSAI


 39%|████████████████████████████████████████████▎                                                                    | 6429/16415 [00:00<00:00, 18618.07it/s]

_ak_update_us_daily throw ex: 'date', symbol = NYX
_ak_update_us_daily throw ex: 'date', symbol = BMAQU
_ak_update_us_daily throw ex: 'date', symbol = OTMO


 52%|███████████████████████████████████████████████████████████▍                                                      | 8565/16415 [00:01<00:01, 7039.84it/s]

...

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋| 16366/16415 [15:56<00:22,  2.22it/s]

_ak_update_us_daily throw ex: 'date', symbol = YXR


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉| 16399/16415 [16:15<00:07,  2.05it/s]

_ak_update_us_daily throw ex: 'date', symbol = ZRSP


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16415/16415 [16:22<00:00, 16.70it/s]


In [126]:
stock_us_daily_df = ak.stock_us_daily(symbol="AAPL", adjust="qfq-factor")

In [132]:
stock_us_daily_df['qfq_factor'] = stock_us_daily_df['qfq_factor'].astype(np.float64)
stock_us_daily_df['adjust'] = stock_us_daily_df['adjust'].astype(np.float64)

In [134]:
stock_us_daily_df

,date,qfq_factor,adjust
0,2025-08-11,1.000000,0.000000
1,2025-05-12,1.000000,-0.260000
2,2025-02-10,1.000000,-0.520000
3,2024-11-11,1.000000,-0.770000
4,2024-11-08,1.000000,-1.020000
...,...,...,...
90,1987-11-17,0.008929,-9.679669
91,1987-08-10,0.008929,-9.679695
92,1987-06-16,0.008929,-9.679714
93,1987-05-11,0.004464,-9.679714


## 3. tushare

- 单次最大8000行，可根据日期循环获取
- 单独付费，2000元/年

In [105]:
dc = DataCenter()

In [110]:
dc.pro.us_daily(ts_code='AAPL')

,ts_code,trade_date,close,open,high,low,pre_close,pct_change,vol,amount,vwap
0,AAPL,20251107,268.47,269.79,272.29,266.77,269.77,-0.48,48227364,1.297508e+10,269.04
1,AAPL,20251106,269.77,267.89,273.40,267.89,270.14,-0.14,51204047,1.386800e+10,270.84
2,AAPL,20251105,270.14,268.61,271.70,266.93,270.04,0.04,43683071,1.178450e+10,269.77
3,AAPL,20251104,270.04,268.32,271.49,267.62,269.05,0.37,49274848,1.329625e+10,269.84
4,AAPL,20251103,269.05,270.42,270.85,266.25,270.37,-0.49,50194583,1.345499e+10,268.06
...,...,...,...,...,...,...,...,...,...,...,...
7995,AAPL,19940202,33.00,33.25,33.25,32.50,33.25,-0.75,1307600,NaN,NaN
7996,AAPL,19940201,33.25,33.00,33.50,32.25,32.75,1.53,1399300,NaN,NaN
7997,AAPL,19940131,32.75,33.50,33.75,32.75,34.00,-3.68,2128400,NaN,NaN
7998,AAPL,19940128,34.00,34.25,34.75,33.75,34.13,-0.37,1218200,NaN,NaN


In [112]:
dc.pro.us_adjfactor(ts_code='AAPL')

Exception: 抱歉，您没有接口访问权限，权限的具体详情访问：https://tushare.pro/document/1?doc_id=108。

In [44]:
spot_em = ak.stock_us_spot_em()

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [43]:
from massive import RESTClient

client = RESTClient("10uFpzT9vu7thJsykdIMrYalPPnxSn4_")

aggs = []
for a in client.list_aggs(
    "AAPL",
    1,
    "day",
    "2025-10-01",
    "2025-10-31",
    limit=50000,
):
    aggs.append(a)

print(aggs)

[Agg(open=255.04, high=258.79, low=254.93, close=255.45, volume=48713940.0, vwap=256.004, timestamp=1759291200000, transactions=535581, otc=None), Agg(open=256.575, high=258.18, low=254.15, close=257.13, volume=42630239.0, vwap=256.855, timestamp=1759377600000, transactions=485229, otc=None), Agg(open=254.665, high=259.24, low=253.95, close=258.02, volume=49155614.0, vwap=257.8596, timestamp=1759464000000, transactions=706052, otc=None), Agg(open=257.99, high=259.07, low=255.05, close=256.69, volume=44664118.0, vwap=256.8692, timestamp=1759723200000, transactions=649161, otc=None), Agg(open=256.805, high=257.4, low=255.43, close=256.48, volume=31955776.0, vwap=256.4092, timestamp=1759809600000, transactions=511382, otc=None), Agg(open=256.52, high=258.52, low=256.11, close=258.06, volume=36496895.0, vwap=257.8119, timestamp=1759896000000, transactions=395556, otc=None), Agg(open=257.805, high=258, low=253.14, close=254.04, volume=38322012.0, vwap=254.5102, timestamp=1759982400000, tran